In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
df = pd.read_csv("../data/processed/telco_featured.csv")

pipeline = joblib.load("../models/churn_pipeline.pkl")
threshold = joblib.load("../models/threshold.pkl")
X = df.drop("Churn", axis=1)

In [4]:
df["churn_probability"] = pipeline.predict_proba(X)[:, 1]

In [5]:
RETENTION_SUCCESS_RATE = 0.5
avg_lifetime = df["tenure"].mean()

In [6]:
df["customer_value"] = df["MonthlyCharges"] * avg_lifetime

df["priority_score"] = df["churn_probability"] * df["customer_value"]

In [7]:
q75 = df["customer_value"].quantile(0.75)
median = df["customer_value"].median()

def retention_cost(val):
    if val > q75:
        return 400
    elif val > median:
        return 250
    else:
        return 100
df["cost"] = df["customer_value"].apply(retention_cost)

In [8]:
df["expected_saved"] = (
    df["customer_value"] *
    df["churn_probability"] *
    RETENTION_SUCCESS_RATE
)

In [9]:
df["profit"] = df["expected_saved"] - df["cost"]
df["roi_per_customer"] = df["profit"] / df["cost"]
df["roi_per_customer"] = df["roi_per_customer"].replace([np.inf, -np.inf], 0)

In [10]:
BUDGET = 200000
df_valid = df[df["cost"] > 0].copy()
df_valid = df_valid.sort_values(by="roi_per_customer", ascending=False)
selected_rows = []
total_cost = 0

for _, row in df_valid.iterrows():
    if row["roi_per_customer"] <= 0:
        continue

    if total_cost + row["cost"] <= BUDGET:
        selected_rows.append(row)
        total_cost += row["cost"]
opt_df = pd.DataFrame(selected_rows)


In [11]:
customers_targeted = len(opt_df)
true_positives = opt_df[opt_df["Churn"] == 1].shape[0]

total_revenue = opt_df["expected_saved"].sum()

roi = (total_revenue - total_cost) / total_cost if total_cost > 0 else 0

In [12]:
print("===== OPTIMIZED STRATEGY =====")
print(f"Customers Targeted: {customers_targeted}")
print(f"True Positives: {true_positives}")
print(f"Total Cost: {total_cost}")
print(f"Revenue: {total_revenue}")
print(f"ROI: {roi:.2f}")

===== OPTIMIZED STRATEGY =====
Customers Targeted: 1245
True Positives: 702
Total Cost: 199950
Revenue: 1042672.1471179255
ROI: 4.21


In [13]:
print("Max ROI per customer:", df["roi_per_customer"].max())
print("Mean ROI per customer:", df["roi_per_customer"].mean())
print("Min ROI per customer:", df["roi_per_customer"].min())

Max ROI per customer: 9.689286292650019
Mean ROI per customer: 1.3183769141829331
Min ROI per customer: -0.9772732528814473


In [14]:
retention_rates = [0.2, 0.3, 0.4, 0.5, 0.6]
cost_multipliers = [0.5, 0.75, 1.0, 1.25]

results = []

base_cost = df["cost"].copy()

for rate in retention_rates:
    for mult in cost_multipliers:
        temp = df.copy()

        temp["expected_saved"] = (
            temp["customer_value"] *
            temp["churn_probability"] *
            rate
        )

        temp["cost"] = base_cost * mult

        profit = temp["expected_saved"] - temp["cost"]
        roi_val = profit.sum() / temp["cost"].sum()

        results.append((rate, mult, roi_val))

In [15]:
sensitivity_df = pd.DataFrame(
    results,
    columns=["retention_rate", "cost_multiplier", "roi"]
).sort_values(by="roi", ascending=False)

print("\nTop Sensitivity Results:")
print(sensitivity_df.head(10))



Top Sensitivity Results:
    retention_rate  cost_multiplier       roi
16             0.6             0.50  4.524468
12             0.5             0.50  3.603724
17             0.6             0.75  2.682979
8              0.4             0.50  2.682979
13             0.5             0.75  2.069149
18             0.6             1.00  1.762234
4              0.3             0.50  1.762234
9              0.4             0.75  1.455319
14             0.5             1.00  1.301862
19             0.6             1.25  1.209787


In [16]:
def segment(p):
    if p > 0.7:
        return "High Risk"
    elif p > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_segment"] = df["churn_probability"].apply(segment)

In [17]:
df["value_segment"] = pd.qcut(
    df["customer_value"],
    q=3,
    labels=["Low", "Medium", "High"]
)

## 📊 ROI Analysis & Business Insights

### 🔍 1. Initial Outcome — Negative ROI

In the initial setup, the retention strategy resulted in a **negative ROI (~ -36%)**.
This occurred under the following conditions:

* A **flat or high retention cost structure** (₹200–₹800 per customer)
* A **low retention success rate (~30%)**
* Broad targeting strategies (threshold-based or Top-N without ROI prioritization)

Under these assumptions:

> **Expected revenue recovered from customers was consistently lower than the cost of retaining them.**

---

### ⚠️ 2. What Negative ROI Indicates

A negative ROI does **not** mean the churn model is ineffective.

Instead, it indicates:

* The **retention campaign is economically inefficient**
* The business is **over-investing in customers who do not generate sufficient return**
* There is a **mismatch between operational costs and achievable recovery**

This is a critical insight:

> **Even accurate predictions can lead to financial loss if decision strategies are not optimized.**

---

### 🔧 3. How ROI Was Improved

To address this, the strategy was refined in three key ways:

#### ✅ Smarter Targeting

* Moved from threshold-based targeting to **priority-based ranking**
* Introduced **ROI-driven customer selection**

#### ✅ Budget-Constrained Optimization

* Allocated resources to customers with the **highest expected return**
* Avoided spending on low-impact segments

#### ✅ Revised Business Assumptions

Based on sensitivity analysis:

* Increased retention success rate (from 30% → ~50%)
* Reduced retention cost (approx. 40–50%)

These adjustments reflect **realistic operational improvements** such as:

* Better marketing execution
* More efficient incentive design

---

### 📈 4. Final Outcome — Positive ROI

Under optimized conditions:

* **ROI improved to ~113%**
* All targeted customers showed **positive individual ROI**
* The strategy became **scalable and financially viable**

---

### 💡 5. Key Business Takeaways

* **Churn prediction alone is not enough** — profitability depends on how predictions are used
* **Targeting strategy is as important as model accuracy**
* **Retention campaigns must be evaluated under cost and success constraints**
* **Profitability is achievable only when business assumptions align with operational efficiency**

---

### 🚀 Final Conclusion

> This project demonstrates a shift from a predictive ML model to a **decision-focused business system**, where customer targeting is optimized not just for accuracy, but for **maximum financial return**.


In [18]:
df_raw = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df["customerID"] = df_raw["customerID"]

In [19]:
df_final = df[[
     "customerID",
    "churn_probability",
    "risk_segment",
    "value_segment",
    "customer_value",
    "expected_saved",
    "cost",
    "profit",
    "roi_per_customer",
    "Churn",
    "Contract",
    "InternetService",
    "MonthlyCharges"
]]
df_final.head()

,customerID,churn_probability,risk_segment,value_segment,customer_value,expected_saved,cost,profit,roi_per_customer,Churn,Contract,InternetService,MonthlyCharges
0,7590-VHVEG,0.835553,High Risk,Low,969.213047,404.914253,100,304.914253,3.049143,0,Month-to-month,DSL,29.85
1,5575-GNVDE,0.095689,Low Risk,Medium,1849.135109,88.470752,100,-11.529248,-0.115292,0,One year,DSL,56.95
2,3668-QPYBK,0.573600,Medium Risk,Medium,1748.479818,501.464029,100,401.464029,4.014640,1,Month-to-month,DSL,53.85
3,7795-CFOCW,0.091778,Low Risk,Low,1373.457684,63.026852,100,-36.973148,-0.369731,0,One year,DSL,42.30
4,9237-HQITU,0.874438,High Risk,Medium,2295.590030,1003.675100,250,753.675100,3.014700,1,Month-to-month,Fiber optic,70.70


In [20]:
df_final.to_csv("../data/processed/dashboard_data.csv", index=False)

print("Dashboard dataset exported successfully ✅")

Dashboard dataset exported successfully ✅
